# Machine Unlearning Robustness Pipeline (Kaggle)

This unified notebook allows you to test the robustness of Machine Unlearning under Quantization right here in Kaggle, utilizing the provided GPU accelerators (T4 recommended).

## 1. Environment Initialization
We first need to install the project dependencies, including `bitsandbytes` for 4-bit/8-bit quantization and Hugging Face `accelerate`.

In [1]:
#!pip install -q -U torch transformers datasets bitsandbytes accelerate rouge-score

#!pip uninstall -y torch torchvision torchaudio
!pip uninstall -y transformers accelerate bitsandbytes

# 2. Install the specific Gemma-3 supported branch
!pip install git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
!pip install git+https://github.com/huggingface/transformers
!pip install --no-cache-dir --upgrade --index-url https://download.pytorch.org/whl/cu118 torch torchvision torchaudio
!pip install -q -U datasets bitsandbytes accelerate rouge-score

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
  Cloning https://github.com/huggingface/transformers (to revision v4.49.0-Gemma-3) to /tmp/pip-req-build-fw9d8295
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-fw9d8295
  Running command git checkout -q 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Resolved https://github.com/huggingface/transformers to commit 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.4 MB/s eta 0:00:00:00:01
  Created wheel for tr

In [2]:
import os
os.chdir(r'/kaggle/working/')

In [3]:
# --- Configuration ---
MODELS_DIR = "models"

# Flags (Set to True to force re-execution even if checkpoints exist)
FLAGS = {
    "RETRAIN_FINE_TUNE_RAW": False,
    "RETRAIN_FINE_TUNE_QA": False,
    "RETRAIN_TASK_VECTOR": True,
    "RETRAIN_GRADIENT_ASCENT": True,
    "REQUANTIZE": True,
    "SHORTEN_TV_TRAIN": True # Shorten task vector train time for testing
}

CHECKPOINTS = {
    "FT_FORGET": os.path.join(MODELS_DIR, "fine_tuned_forget"),
    "FT_FORGET_QA": os.path.join(MODELS_DIR, "fine_tuned_forget_qa"),
    "TV_UNLEARN": os.path.join(MODELS_DIR, "unlearned_task_vector"),
    "TV_UNLEARN_QA": os.path.join(MODELS_DIR, "unlearned_task_vector_qa"),
    "TV_UNLEARN_BOTH": os.path.join(MODELS_DIR, "unlearned_task_vector_both"),
    "GA_UNLEARN": os.path.join(MODELS_DIR, "unlearned_gradient_ascent"),
}

In [4]:
# import os
# import shutil

# # --- MODEL TRANSFER FROM KAGGLE INPUT TO OUTPUT ---
# # IMPORTANT: Replace 'YOUR_DATASET_NAME' with the actual folder name in Kaggle's input panel
# INPUT_SOURCE_DIR = "/kaggle/input/models/anurag2006/initial-tv-models/transformers/default/1/models" 
# OUTPUT_MODELS_DIR = MODELS_DIR # Usually resolves to /kaggle/working/models

# print(f"Checking for input models in: {INPUT_SOURCE_DIR}")

# if os.path.exists(INPUT_SOURCE_DIR):
#     # Ensure the output models directory exists
#     os.makedirs(OUTPUT_MODELS_DIR, exist_ok=True)
    
#     print(f"Transferring models to {OUTPUT_MODELS_DIR}...")
    
#     # Iterate through and copy everything from the input folder
#     for item in os.listdir(INPUT_SOURCE_DIR):
#         source_path = os.path.join(INPUT_SOURCE_DIR, item)
#         dest_path = os.path.join(OUTPUT_MODELS_DIR, item)
        
#         if os.path.isdir(source_path):
#             print(f" -> Merging/Replacing directory: {item}")
#             # dirs_exist_ok=True allows it to overwrite/merge with existing folders seamlessly
#             shutil.copytree(source_path, dest_path, dirs_exist_ok=True)
#         else:
#             print(f" -> Copying file: {item}")
#             shutil.copy2(source_path, dest_path)
            
#     print("✅ Transfer complete! Older models are now ready for inference/evaluation.")
# else:
#     print(f"⚠️ Input directory '{INPUT_SOURCE_DIR}' not found.")
#     print("If you haven't attached the dataset/model to this notebook yet, use the 'Add Data' button on the right sidebar.")

### Hugging Face Login
Gemma requires authentication. Ensure you have added your Hugging Face token as `HF_TOKEN` inside the **Kaggle Secrets** add-on menu.

In [5]:
from kaggle_secrets import UserSecretsClient
import huggingface_hub

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("\n--- Authenticated with Hugging Face successfully! ---")
except Exception as e:
    print("\n--- Warning: Could not find HF_TOKEN in Kaggle Secrets ---")
    print("Please set it up via Add-ons -> Secrets to download Gemma.\n", e)


--- Authenticated with Hugging Face successfully! ---


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from torch.optim import AdamW
from rouge_score import rouge_scorer

MODEL_NAME = "google/gemma-3-1b-it"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 2. Model and Dataset Operations

In [7]:
def load_base_model():
    print(f"Loading Base Model {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    return model, tokenizer

def load_muse_data():
    """
    Loads the MUSE-Books dataset.
    - 'raw' config for training (text unlearning)
    - 'knowmem' config for evaluation (QA pairs)
    """
    print("Loading MUSE-Books dataset...")
    
    # 1. Load Raw Text (for Unlearning / Fine-tuning)
    # Splits: 'forget', 'retain1' (Retain), 'holdout'
    print("Loading Raw Text (subset='raw')...")
    try:
        raw_dataset = load_dataset("muse-bench/MUSE-Books", "raw")
    except ValueError:
        print("Warning: 'raw' config not found or dataset issue. Fallback to default...")
        raw_dataset = load_dataset("muse-bench/MUSE-Books")

    # Inspect available splits 
    print(f"Available 'raw' splits: {raw_dataset.keys()}")
    forget_train = raw_dataset['forget']
    # retain1 is the standard retention set name in MUSE-Books raw
    retain_train = raw_dataset['retain1'] if 'retain1' in raw_dataset else raw_dataset['retain']
    
    # 2. Load QA Pairs (for Evaluation)
    print("Loading Knowledge Memorization (subset='knowmem')...")
    try:
        qa_dataset = load_dataset("muse-bench/MUSE-Books", "knowmem")
        print(f"Available 'knowmem' splits: {qa_dataset.keys()}")
        
        # Identifying correct QA splits
        # Usually 'forget_qa' or 'forget'
        qa_forget_key = next((k for k in ['forget_qa', 'forget'] if k in qa_dataset), None)
        qa_retain_key = next((k for k in ['retain_qa', 'retain', 'retain1'] if k in qa_dataset), None)

        if qa_forget_key and qa_retain_key:
            forget_qa = qa_dataset[qa_forget_key]
            retain_qa = qa_dataset[qa_retain_key]
        else:
             # Fallback if specific keys aren't found
            print("Warning: Specific QA keys not found. Using available splits.")
            forget_qa = qa_dataset[list(qa_dataset.keys())[0]]
            retain_qa = qa_dataset[list(qa_dataset.keys())[1]] if len(qa_dataset.keys()) > 1 else forget_qa

    except Exception as e:
        print(f"Warning: Could not load knowmem subset ({e}). Using raw text as fallback for evaluation (suboptimal).")
        forget_qa = forget_train
        retain_qa = retain_train

    print(f"Loaded Training Data: {len(forget_train)} forget samples, {len(retain_train)} retain samples.")
    print(f"Loaded Evaluation Data: {len(forget_qa)} forget QA pairs, {len(retain_qa)} retain QA pairs.")
    
    return forget_train, retain_train, forget_qa, retain_qa

## 3. Unlearning Implementations
These functions define Task Arithmetic and Gradient Ascent (Baseline NPO methods).

In [8]:
def compute_task_vector(pretrained_model, finetuned_model):
    task_vector = {}
    # Use the state_dict for easier access
    ft_state = finetuned_model.state_dict()
    
    for name, base_param in pretrained_model.named_parameters():
        if name in ft_state:
            # Move the fine-tuned parameter to the same device as the base parameter
            ft_param = ft_state[name].to(base_param.device)
            task_vector[name] = ft_param - base_param
            
    return task_vector

def apply_task_vector(pretrained_model, task_vector, scaling_factor):
    # Create a copy so we don't modify the original base model in memory
    import copy
    unlearned_model = copy.deepcopy(pretrained_model)
    
    with torch.no_grad():
        for name, param in unlearned_model.named_parameters():
            if name in task_vector:
                # Move the task vector tensor to the same device as this specific layer
                vector_component = task_vector[name].to(param.device)
                
                # Perform the scaled subtraction/addition
                param.add_(vector_component * scaling_factor)
                
    return unlearned_model

# def gradient_ascent_unlearning(model, forget_dataloader, epochs=1, lr=1e-5):
#     print("Executing standard Gradient Ascent...")
#     model.train()
#     optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
#     # Get the device of the model (handles multi-GPU safely)
#     device = next(model.parameters()).device 
    
#     for epoch in range(epochs):
#         for batch in forget_dataloader:
#             # Move all tensors to the model's primary device
#             inputs = batch['input_ids'].to(device)
#             masks = batch['attention_mask'].to(device)
#             labels = batch['labels'].to(device)
            
#             outputs = model(input_ids=inputs, attention_mask=masks, labels=labels)
            
#             # Gradient ascent: we want to MAXIMIZE loss (negative loss)
#             loss = -outputs.loss 
#             loss.backward()
#             optimizer.step()
#             optimizer.zero_grad()
            
#     return model

In [9]:
import torch.nn.functional as F

def get_batch_logps(logits, labels):
    """Calculates the sequence-level log probabilities of the true labels."""
    # Shift so that tokens < n predict n
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    # Calculate cross entropy per token
    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    token_log_probs = -loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    token_log_probs = token_log_probs.view(shift_labels.size())
    
    # Mask out padding (-100) and sum over the sequence
    mask = (shift_labels != -100)
    seq_log_probs = (token_log_probs * mask).sum(dim=-1)
    
    return seq_log_probs

def npo_unlearning(model, ref_model, forget_dataloader, epochs=1, lr=1e-5, beta=0.1):
    print("Executing Negative Preference Optimization (NPO)...")
    model.train()
    ref_model.eval() # The reference model must remain frozen
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    # Use generic GradScaler if cuda.amp is not available or for better compatibility
    scaler = torch.cuda.amp.GradScaler()
    device = model.device
    
    for epoch in range(epochs):
        for step, batch in enumerate(forget_dataloader):
            inputs = batch['input_ids'].to(device)
            masks = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                # 1. Get policy model (current model) log probabilities
                outputs = model(input_ids=inputs, attention_mask=masks)
                policy_logps = get_batch_logps(outputs.logits, labels)
                
                # 2. Get reference model (base model) log probabilities
                with torch.no_grad():
                    ref_outputs = ref_model(input_ids=inputs, attention_mask=masks)
                    ref_logps = get_batch_logps(ref_outputs.logits, labels)
                
                # 3. Calculate NPO Loss: -(2/beta) * log_sigmoid(-beta * (policy_logps - ref_logps))
                ratio = policy_logps - ref_logps
                
                # Clamp ratio to prevent instability if models diverge too much
                # If policy >>> ref, ratio is huge positive -> -beta*ratio is huge negative -> logsigmoid is linear
                # If policy <<< ref, ratio is huge negative -> -beta*ratio is huge positive -> logsigmoid is 0
                
                loss = -(2.0 / beta) * F.logsigmoid(-beta * ratio).mean()
            
            # Check for NaNs before backward
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"Skipping batch {step}: loss is NaN/Inf")
                continue
            
            # Scaled Backward
            scaler.scale(loss).backward()
            
            # Unscale for clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            # Cleanup to prevent OOM
            if step % 10 == 0:
                torch.cuda.empty_cache()
            
    return model

In [10]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

def tokenize_dataset(dataset, tokenizer, max_length=512):
    """
    Tokenizes the dataset for training/unlearning by chunking into max_length.
    Instead of truncating to 512 tokens (which only uses ~1KB of text), 
    we chunk all text into 512-token blocks to maximize training data usage.
    """
    # Identify the text column
    text_col = 'text' if 'text' in dataset.column_names else dataset.column_names[0]
    
    # 1. Tokenize all texts
    tokenized_dataset = dataset.map(
        lambda x: tokenizer(x[text_col]),
        batched=True,
        remove_columns=dataset.column_names
    )
    
    # 2. Chunking Logic (Group text into blocks of max_length)
    def group_texts(examples):
        concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        
        # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can customize this part to your needs.
        if total_length >= max_length:
            total_length = (total_length // max_length) * max_length
            
        result = {
            k: [t[i : i + max_length] for i in range(0, total_length, max_length)]
            for k, t in concatenated_examples.items()
        }
        result["labels"] = result["input_ids"].copy()
        return result

    lm_dataset = tokenized_dataset.map(group_texts, batched=True)
    lm_dataset.set_format("torch")
    
    print(f"Dataset chunked from {len(dataset)} examples to {len(lm_dataset)} blocks.")
    return lm_dataset

def create_dataloader(dataset, tokenizer, batch_size=4):
    tokenized = tokenize_dataset(dataset, tokenizer)
    # The collator automatically creates 'labels' from 'input_ids'
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    return DataLoader(
        tokenized, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=data_collator # Add this line
    )

from peft import LoraConfig, get_peft_model

def fine_tune_model(model, tokenizer, dataset, output_dir, epochs=5, is_qa=False):
    """
    Fine-tunes the model. Uses chunking for raw text, but independent padding for QA.
    """
    lora_config = LoraConfig(
        r=8, 
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    
    print(f"Fine-tuning model on {len(dataset)} samples...")
    
    if is_qa:
        # QA Tokenization: Treat every row independently (NO chunking)
        def tokenize_qa(examples):
            # Max length 1024 is usually plenty for Q&A
            return tokenizer(examples['text'], truncation=True, max_length=1024)
        
        tokenized_dataset = dataset.map(tokenize_qa, batched=True, remove_columns=dataset.column_names)
        print(f"QA Dataset structured as {len(tokenized_dataset)} independent sequences.")
    else:
        # Raw Text Tokenization: Use your existing chunking function
        tokenized_dataset = tokenize_dataset(dataset, tokenizer)
        
    # --- REPLACE THIS BLOCK INSIDE fine_tune_model IN CELL 33 ---
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=4,   # Increased from 2 to use more VRAM
        gradient_accumulation_steps=4,  # Adjusted to keep effective batch size consistent
        learning_rate=5e-5,             # Slightly higher for faster convergence in few epochs
        weight_decay=0.01,
        save_strategy="no", 
        logging_steps=10,
        fp16=True,                      # CRITICAL: Enabled for T4 speedup
        bf16=False,
        report_to="none",
        optim="paged_adamw_8bit" 
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )
    
    trainer.train()
    return model

def prepare_cloze_questions(dataset):
    """
    Attempts to prepare cloze questions from the dataset for evaluation.
    Expects 'question' and 'answer' columns, or similar.
    """
    questions = []
    # Check for common QA column names
    q_col = next((col for col in ['question', 'prompt', 'input'] if col in dataset.column_names), None)
    a_col = next((col for col in ['answer', 'target', 'output'] if col in dataset.column_names), None)
    
    if q_col and a_col:
        for item in dataset:
            questions.append({
                'prompt': item[q_col],
                'answer': item[a_col]
            })
    else:
        # Fallback: create dummy tasks or just warn
        print("Warning: Could not identify question/answer columns for Cloze Evaluation. Evaluation might be skipped or inaccurate.")
        # If dataset is text only, maybe treating start as prompt?
        # For valid unlearning evaluation, we usually need specific queries. 
        # Using first 50 chars as prompt and next 10 as answer (naive)
        if 'text' in dataset.column_names:
            for item in dataset.select(range(min(50, len(dataset)))):
                text = item['text']
                if len(text) > 100:
                    questions.append({
                        'prompt': text[:50],
                        'answer': text[50:60] # Naive
                    })
    
    return questions

def prepare_raw_text_continuation(dataset, num_samples=50):
    """
    Creates prompt-answer pairs strictly from the raw textual dataset.
    Takes the first 30 words as a prompt, and the consecutive 20 words as the target answer.
    This directly measures "verbatim memorization" of the raw text.
    """
    pairs = []
    text_col = 'text' if 'text' in dataset.column_names else dataset.column_names[0]
    subset = dataset.select(range(min(num_samples, len(dataset))))
    
    for item in subset:
        text = item[text_col]
        words = text.split()
        if len(words) >= 50:
            prompt = " ".join(words[:30])
            answer = " ".join(words[30:50])
            pairs.append({'prompt': prompt, 'answer': answer})
            
    return pairs

## 4. Post-Training Quantization (PTQ)
Loading the unlearned FP16 models in 4-bit to observe the Recovery Delta.

In [11]:
def load_quantized_model(model_dir_path, bit_width=4):
    print(f"Quantizing model from {model_dir_path} to {bit_width}-bit precision...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=(bit_width == 4),
        load_in_8bit=(bit_width == 8),
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    
    quantized_model = AutoModelForCausalLM.from_pretrained(
        model_dir_path,
        device_map="auto",
        quantization_config=bnb_config
    )
    return quantized_model

## 5. Evaluation Loop

In [12]:
import math
import torch
from rouge_score import rouge_scorer

def evaluate_rouge_score(model, tokenizer, cloze_questions):
    if not cloze_questions:
        return 0.0
        
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    total_rouge_l_f1 = 0.0
    
    for question_dict in cloze_questions:
        prompt = question_dict['prompt']
        target_answer = question_dict['answer'].lower()
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_length = inputs["input_ids"].shape[1] # Track exact token length
        
        outputs = model.generate(**inputs, max_new_tokens=30, pad_token_id=tokenizer.eos_token_id)
        
        # Slice the tokens directly before decoding
        new_tokens = outputs[0][input_length:]
        continuation = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        
        score = scorer.score(target_answer, continuation)
        total_rouge_l_f1 += score['rougeL'].fmeasure
            
    return total_rouge_l_f1 / len(cloze_questions)

def evaluate_perplexity(model, tokenizer, dataset, num_samples=50):
    """
    Calculates the perplexity of the model on a subset of the dataset.
    High perplexity = model is "surprised" by the text (good for forget set).
    Low perplexity = model predicts the text well (good for retain set).
    """
    if not dataset or len(dataset) == 0:
        return float('inf')
        
    text_col = 'text' if 'text' in dataset.column_names else dataset.column_names[0]
    subset = dataset.select(range(min(num_samples, len(dataset))))
    
    total_loss = 0.0
    total_length = 0
    
    model.eval()
    with torch.no_grad():
        for item in subset:
            text = item[text_col]
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
            
            if inputs['input_ids'].size(1) < 2:
                continue
                
            outputs = model(
                input_ids=inputs['input_ids'], 
                attention_mask=inputs['attention_mask'], 
                labels=inputs["input_ids"]
            )
            
            # Causal LM loss is calculated over N-1 tokens
            valid_seq_len = inputs['input_ids'].size(1) - 1 
            
            total_loss += outputs.loss.item() * valid_seq_len
            total_length += valid_seq_len
            
    if total_length == 0:
        return float('inf')
        
    avg_loss = total_loss / total_length
    
    if avg_loss > 20: 
        return float('inf')
        
    return math.exp(avg_loss)

# --- ADD TO THE BOTTOM OF CELL 35 ---
def run_robust_evaluation(model_path, model_name, tokenizer, eval_tasks, is_quantized=False):
    print(f"\n--- Evaluating: {model_name} ---")
    clean_memory() # Clear leftovers from previous runs
    
    # Configure 4-bit loading if requested
    bnb_config = None
    if is_quantized:
        from transformers import BitsAndBytesConfig
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

    # Load the specific variant
    model = AutoModelForCausalLM.from_pretrained(
        model_path, 
        quantization_config=bnb_config, 
        torch_dtype=torch.float16, 
        device_map="auto"
    )

    # Calculate metrics
    metrics = {
        "Model": model_name,
        "Forget QA ROUGE": evaluate_rouge_score(model, tokenizer, eval_tasks['forget_qa_q']),
        "Retain QA ROUGE": evaluate_rouge_score(model, tokenizer, eval_tasks['retain_qa_q']),
        "Forget Raw ROUGE": evaluate_rouge_score(model, tokenizer, eval_tasks['forget_raw_p']),
        "Forget PPL": evaluate_perplexity(model, tokenizer, eval_tasks['forget_train'], num_samples=30),
    }

    # CRITICAL: Wipe from GPU before next model loads
    del model
    clean_memory()
    return metrics

## Execution Sandbox
Run your components below to generate the metrics outlined in your proposal.

In [13]:
import os
import shutil
import torch
import pandas as pd
from transformers import AutoModelForCausalLM
import gc

os.makedirs(MODELS_DIR, exist_ok=True)

# Helper function to clear memory aggressively
def clean_memory():
    gc.collect()
    torch.cuda.empty_cache()

# --- Execution Pipeline ---

print("\n=== PHASE 1: Loading Resources ===")
# Load Data
forget_train, retain_train, forget_qa, retain_qa = load_muse_data()

# Prepare Evaluation Data

# 1. QA Conceptual Knowledge Evaluation
forget_qa_questions = prepare_cloze_questions(forget_qa.select(range(min(50, len(forget_qa)))))
retain_qa_questions = prepare_cloze_questions(retain_qa.select(range(min(50, len(retain_qa)))))
print(f"Prepared {len(forget_qa_questions)} QA forget questions and {len(retain_qa_questions)} QA retain questions.")

# 2. Raw Text Verbatim Memorization Evaluation
forget_raw_pairs = prepare_raw_text_continuation(forget_train, num_samples=50)
retain_raw_pairs = prepare_raw_text_continuation(retain_train, num_samples=50)
print(f"Prepared {len(forget_raw_pairs)} Raw forget completions and {len(retain_raw_pairs)} Raw retain completions.")

# Load Tokenizer (Base model needed momentarily for tokenizer if not separate)
print("Loading Base Model Tokenizer...")
_, tokenizer = load_base_model()
del _ # Unload model if loaded, keep tokenizer
clean_memory()

# --- PHASE 2: Fine-Tuning (Task Vector Prep) ---
print("\n=== PHASE 2: Fine-Tuning (Task Vector Prep) ===")
if not os.path.exists(CHECKPOINTS["FT_FORGET"]) or FLAGS["RETRAIN_FINE_TUNE_RAW"]:
    print("Starting Fine-Tuning on Forget Set (Raw Text)...")
    base_model, _ = load_base_model() # Load fresh
    
    tv_epochs = 2 if FLAGS.get("SHORTEN_TV_TRAIN", False) else 10
    print(f"Task Vector Fine-Tuning Epochs set to: {tv_epochs}")
    ft_model = fine_tune_model(base_model, tokenizer, forget_train, CHECKPOINTS["FT_FORGET"], epochs=tv_epochs)
    
    print(f"Saving Fine-Tuned Model to {CHECKPOINTS['FT_FORGET']}...")
    ft_model.save_pretrained(CHECKPOINTS["FT_FORGET"])
    tokenizer.save_pretrained(CHECKPOINTS["FT_FORGET"])
    
    del base_model, ft_model
    clean_memory()
else:
    print(f"Checkpoint found: {CHECKPOINTS['FT_FORGET']}")

# --- PHASE 2.1: Fine-Tuning (Task Vector Prep - QA) ---
print("\n=== PHASE 2.1: Fine-Tuning (Task Vector Prep - QA) ===")
if not os.path.exists(CHECKPOINTS["FT_FORGET_QA"]) or FLAGS["RETRAIN_FINE_TUNE_QA"]:
    print("Starting Fine-Tuning on Forget Set (QA)...")
    base_model, _ = load_base_model() # Load fresh
    
    def format_qa_for_training(example):
        q_cols = ['question', 'prompt', 'input']
        a_cols = ['answer', 'target', 'output']
        q_col = next((col for col in q_cols if col in example.keys()), None)
        a_col = next((col for col in a_cols if col in example.keys()), None)
        if q_col and a_col:
            # Gemma 3 chat template
            messages = [
                {"role": "user", "content": example[q_col]},
                {"role": "assistant", "content": example[a_col]}
            ]
            formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
            return {"text": formatted_text}
        return {"text": str(example)}
        
    forget_qa_train = forget_qa.map(format_qa_for_training)
    
    # Increase QA epochs since the dataset is small. 15-20 is a good target for strong overfitting.
    tv_qa_epochs = 5 if FLAGS.get("SHORTEN_TV_TRAIN", False) else 20
    print(f"Task Vector QA Fine-Tuning Epochs set to: {tv_qa_epochs}")
    
    # Notice the is_qa=True flag here!
    ft_model_qa = fine_tune_model(base_model, tokenizer, forget_qa_train, CHECKPOINTS["FT_FORGET_QA"], epochs=tv_qa_epochs, is_qa=True)
    
    print(f"Saving QA Fine-Tuned Model to {CHECKPOINTS['FT_FORGET_QA']}...")
    ft_model_qa.save_pretrained(CHECKPOINTS["FT_FORGET_QA"])
    tokenizer.save_pretrained(CHECKPOINTS["FT_FORGET_QA"])
    
    del base_model, ft_model_qa
    clean_memory()
else:
    print(f"Checkpoint found: {CHECKPOINTS['FT_FORGET_QA']}")

# --- PHASE 3: Task Vector Unlearning ---
print("\n=== PHASE 3: Task Vector Unlearning ===")
if not os.path.exists(CHECKPOINTS["TV_UNLEARN"]) or FLAGS["RETRAIN_TASK_VECTOR"]:
    print("Computing Task Vector...")
    base_model, _ = load_base_model()
    ft_model = AutoModelForCausalLM.from_pretrained(CHECKPOINTS["FT_FORGET"], torch_dtype=torch.float16, device_map="auto")
    
    task_vector = compute_task_vector(base_model, ft_model)
    # Scaling factor logic: (Base + (FineTuned - Base) * -scaling) -> Unlearn
    # Increased magnitude to -1.0 to ensure stronger unlearning
    scaling_factor = -1.0
    tv_unlearned_model = apply_task_vector(base_model, task_vector, scaling_factor=scaling_factor)
    
    print(f"Saving TV Unlearned Model to {CHECKPOINTS['TV_UNLEARN']}...")
    tv_unlearned_model.save_pretrained(CHECKPOINTS["TV_UNLEARN"])
    tokenizer.save_pretrained(CHECKPOINTS["TV_UNLEARN"])
    
    del base_model, ft_model, tv_unlearned_model, task_vector
    clean_memory()
else:
    print(f"Checkpoint found: {CHECKPOINTS['TV_UNLEARN']}")

# --- PHASE 3.1: Task Vector Unlearning (QA) ---
print("\n=== PHASE 3.1: Task Vector Unlearning (QA) ===")
if not os.path.exists(CHECKPOINTS["TV_UNLEARN_QA"]) or FLAGS["RETRAIN_TASK_VECTOR"]:
    print("Computing Task Vector (QA)...")
    base_model, _ = load_base_model()
    ft_model_qa = AutoModelForCausalLM.from_pretrained(CHECKPOINTS["FT_FORGET_QA"], torch_dtype=torch.float16, device_map="auto")
    
    task_vector_qa = compute_task_vector(base_model, ft_model_qa)
    scaling_factor = -1.0 
    tv_unlearned_qa = apply_task_vector(base_model, task_vector_qa, scaling_factor=scaling_factor)
    
    print(f"Saving QA TV Unlearned Model to {CHECKPOINTS['TV_UNLEARN_QA']}...")
    tv_unlearned_qa.save_pretrained(CHECKPOINTS["TV_UNLEARN_QA"])
    tokenizer.save_pretrained(CHECKPOINTS["TV_UNLEARN_QA"])
    
    del base_model, ft_model_qa, tv_unlearned_qa, task_vector_qa
    clean_memory()
else:
    print(f"Checkpoint found: {CHECKPOINTS['TV_UNLEARN_QA']}")

# --- PHASE 3.2: Task Vector Unlearning (Combined Raw + QA) ---
print("\n=== PHASE 3.2: Task Vector Unlearning (Combined Raw + QA) ===")
if not os.path.exists(CHECKPOINTS["TV_UNLEARN_BOTH"]) or FLAGS["RETRAIN_TASK_VECTOR"]:
    print("Computing Combined Task Vector...")
    base_model, _ = load_base_model()
    
    ft_model_raw = AutoModelForCausalLM.from_pretrained(CHECKPOINTS["FT_FORGET"], torch_dtype=torch.float16, device_map="auto")
    tv_raw = compute_task_vector(base_model, ft_model_raw)
    del ft_model_raw
    clean_memory()
    
    ft_model_qa = AutoModelForCausalLM.from_pretrained(CHECKPOINTS["FT_FORGET_QA"], torch_dtype=torch.float16, device_map="auto")
    tv_qa = compute_task_vector(base_model, ft_model_qa)
    del ft_model_qa
    clean_memory()
    
    print("Combining Task Vectors...")
    combined_tv = {k: (tv_raw[k] + tv_qa[k]) / 2.0 for k in tv_raw}
    del tv_raw, tv_qa
    clean_memory()
    
    scaling_factor = -1.0
    tv_unlearned_both = apply_task_vector(base_model, combined_tv, scaling_factor=scaling_factor)
    
    print(f"Saving Combined TV Unlearned Model to {CHECKPOINTS['TV_UNLEARN_BOTH']}...")
    tv_unlearned_both.save_pretrained(CHECKPOINTS["TV_UNLEARN_BOTH"])
    tokenizer.save_pretrained(CHECKPOINTS["TV_UNLEARN_BOTH"])
    
    del base_model, tv_unlearned_both, combined_tv
    clean_memory()
else:
    print(f"Checkpoint found: {CHECKPOINTS['TV_UNLEARN_BOTH']}")

# # --- PHASE 4: NPO Unlearning ---
# print("\n=== PHASE 4: NPO Unlearning ===")
# if True: # Always run NPO for now as per user request to retry/fix
# # if not os.path.exists(CHECKPOINTS["GA_UNLEARN"]) or FLAGS["RETRAIN_GRADIENT_ASCENT"]:
#     print("Executing NPO Unlearning...")
#     # Load the base model to use as the frozen reference
#     ref_model, _ = load_base_model() 
    
#     # Load a fresh copy of the base model to apply unlearning to
#     unlearning_model, _ = load_base_model() 
    
#     # Create DataLoader using new function (chunks text into 512 tokens -> many batches)
#     forget_loader = create_dataloader(forget_train, tokenizer, batch_size=2)
    
#     # Run NPO (beta=0.1)
#     # Increased epochs to 5 for better convergence
#     ga_unlearned_model = npo_unlearning(
#         unlearning_model, 
#         ref_model, 
#         forget_loader, 
#         epochs=5, 
#         lr=1e-5, 
#         beta=0.1
#     )
    
#     print(f"Saving NPO Unlearned Model to {CHECKPOINTS['GA_UNLEARN']}...")
#     ga_unlearned_model.save_pretrained(CHECKPOINTS["GA_UNLEARN"])
#     tokenizer.save_pretrained(CHECKPOINTS["GA_UNLEARN"])
    
#     del ref_model, unlearning_model, ga_unlearned_model
#     clean_memory()
# else:
#     print(f"Checkpoint found: {CHECKPOINTS['GA_UNLEARN']}")

# --- PHASE 5: Evaluation & Quantization ---
print("\n=== PHASE 5: Evaluation & Quantization ===")
results = []

def run_evaluation(model_obj_or_path, model_name, is_quantized=False):
    print(f"Evaluating {model_name}...")
    try:
        model = None
        if isinstance(model_obj_or_path, str):
            if is_quantized:
                # Load bitsandbytes quantized
                try:
                    from transformers import BitsAndBytesConfig
                    bnb_config = BitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_compute_dtype=torch.float16
                    )
                    model = AutoModelForCausalLM.from_pretrained(
                        model_obj_or_path, 
                        quantization_config=bnb_config, 
                        device_map="auto"
                    )
                except Exception as e:
                    print(f"Quantization failed: {e}. Loading FP16 instead.")
                    model = AutoModelForCausalLM.from_pretrained(model_obj_or_path, torch_dtype=torch.float16, device_map="auto")
            else:
                model = AutoModelForCausalLM.from_pretrained(model_obj_or_path, torch_dtype=torch.float16, device_map="auto")
        else:
            model = model_obj_or_path

        forget_qa_rouge = evaluate_rouge_score(model, tokenizer, forget_qa_questions)
        retain_qa_rouge = evaluate_rouge_score(model, tokenizer, retain_qa_questions)
        
        forget_raw_rouge = evaluate_rouge_score(model, tokenizer, forget_raw_pairs)
        retain_raw_rouge = evaluate_rouge_score(model, tokenizer, retain_raw_pairs)
        
        forget_ppl = evaluate_perplexity(model, tokenizer, forget_train, num_samples=30)
        retain_ppl = evaluate_perplexity(model, tokenizer, retain_train, num_samples=30)
        
        # Helper cleanup
        if isinstance(model_obj_or_path, str):
            del model
            clean_memory()
            
        return {
            "Model": model_name,
            "Forget QA ROUGE": forget_qa_rouge,
            "Retain QA ROUGE": retain_qa_rouge,
            "Forget Raw ROUGE": forget_raw_rouge,
            "Retain Raw ROUGE": retain_raw_rouge,
            "Forget PPL": forget_ppl,
            "Retain PPL": retain_ppl
        }
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")
        return {
            "Model": model_name, 
            "Forget QA ROUGE": "Error", 
            "Retain QA ROUGE": "Error",
            "Forget Raw ROUGE": "Error",
            "Retain Raw ROUGE": "Error",
            "Forget PPL": "Error",
            "Retain PPL": "Error"
        }

# 1. Base Model
# print("Evaluating Base Model...")
# base_model, _ = load_base_model()
# results.append(run_evaluation(base_model, "Base Model (FP16)"))
# del base_model
# clean_memory()



# # 2. Unlearned Models (FP16 & 4-bit)
# checkpoints = [
#     ("TV Unlearned (Raw)", CHECKPOINTS["TV_UNLEARN"]),
#     ("TV Unlearned (QA)", CHECKPOINTS["TV_UNLEARN_QA"]),
#     ("TV Unlearned (Combined)", CHECKPOINTS["TV_UNLEARN_BOTH"]),
#     # ("GA Unlearned", CHECKPOINTS["GA_UNLEARN"])
# ]

# for name, path in checkpoints:
#     if os.path.exists(path):
#         # FP16 Evaluation
#         results.append(run_evaluation(path, f"{name} (FP16)", is_quantized=False))
#         # 4-bit Quantization & Evaluation
#         if FLAGS["REQUANTIZE"]:
#              results.append(run_evaluation(path, f"{name} (4-bit)", is_quantized=True))

# # --- Summary ---
# print("\n=== Final Results ===")
# df_results = pd.DataFrame(results)
# print(df_results)
# if not df_results.empty:
#     df_results.to_csv("unlearning_benchmark_results.csv", index=False)
# print("Pipeline Successfully Completed!")

# Package data for the evaluator
tasks = {
    'forget_qa_q': forget_qa_questions,
    'retain_qa_q': retain_qa_questions,
    'forget_raw_p': forget_raw_pairs,
    'forget_train': forget_train
}

# 1. Evaluate Base Model
results.append(run_robust_evaluation("google/gemma-3-1b-it", "Base Model (FP16)", tokenizer, tasks))

# 2. Evaluate Unlearned Variations (FP16 vs 4-bit)
configs = [
    ("TV Raw", CHECKPOINTS["TV_UNLEARN"]),
    ("TV Combined", CHECKPOINTS["TV_UNLEARN_BOTH"])
]

for name, path in configs:
    if os.path.exists(path):
        results.append(run_robust_evaluation(path, f"{name} (FP16)", tokenizer, tasks, is_quantized=False))
        results.append(run_robust_evaluation(path, f"{name} (4-bit)", tokenizer, tasks, is_quantized=True))

# Create final report
df_results = pd.DataFrame(results)
print("\n=== FINAL BENCHMARK RESULTS ===")
print(df_results)
df_results.to_csv("unlearning_benchmark_results.csv", index=False)




=== PHASE 1: Loading Resources ===
Loading MUSE-Books dataset...
Loading Raw Text (subset='raw')...


README.md: 0.00B [00:00, ?B/s]

raw/retain2-00000-of-00001.parquet:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

raw/forget-00000-of-00001.parquet:   0%|          | 0.00/2.43M [00:00<?, ?B/s]

raw/retain1-00000-of-00001.parquet:   0%|          | 0.00/472k [00:00<?, ?B/s]

...


--- Evaluating: TV Combined (FP16) ---


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]


--- Evaluating: TV Combined (4-bit) ---


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]


=== FINAL BENCHMARK RESULTS ===
                 Model  Forget QA ROUGE  Retain QA ROUGE  Forget Raw ROUGE  \
0    Base Model (FP16)         0.058281         0.047295          0.129381   
1        TV Raw (FP16)         0.058307         0.043258          0.121772   
2       TV Raw (4-bit)         0.055642         0.056268          0.144721   
3   TV Combined (FP16)         0.057812         0.060810          0.180361   
4  TV Combined (4-bit)         0.055375         0.028395          0.163538   

   Forget PPL  
0   28.178883  
1   28.178883  
2   32.372967  
3   28.178883  
4   32.372967  


In [16]:
# --- NEW FINAL CELL ---
def calculate_snapback(df):
    """Calculates Δ_Rec for the primary unlearning method."""
    try:
        fp16_score = df.loc[df['Model'] == "TV Raw (FP16)", "Forget QA ROUGE"].values[0]
        bit4_score = df.loc[df['Model'] == "TV Raw (4-bit)", "Forget QA ROUGE"].values[0]
        
        delta_rec = bit4_score - fp16_score
        print(f"Recovery Delta (Δ_Rec): {delta_rec:.5f}")
        
        if delta_rec > 0:
            print(f"⚠️ FAILURE: Knowledge recovered. Rounding error restored {delta_rec*100:.2f}% of data.")
        else:
            print("✅ SUCCESS: Unlearning is robust to 4-bit quantization.")
    except IndexError:
        print("Missing rows in dataframe to calculate Δ_Rec.")

calculate_snapback(df_results)

Recovery Delta (Δ_Rec): -0.00266
✅ SUCCESS: Unlearning is robust to 4-bit quantization.
